# 고급 RAG 시스템 - 최신 ICT 시사용어 2025

이 노트북은 여러 RAG 성능 향상 기법들을 LangGraph로 결합하여 구현합니다.

## 주요 기법
1. **Query Rewriting**: 검색 최적화를 위한 쿼리 재작성
2. **HyDE (Hypothetical Document Embeddings)**: 가상 답변 생성으로 검색 품질 향상
3. **Multi-Query Retrieval**: 다양한 쿼리로 검색 범위 확대
4. **Semantic Reranking**: 검색 결과 재정렬
5. **Self-RAG**: 답변 품질 자체 평가 및 재시도
6. **Multi-turn with Memory**: 대화 이력 관리


## 1. 환경 설정 및 초기화


In [ ]:
from dotenv import load_dotenv
import os
import glob
from typing import TypedDict, Literal, List, Annotated, Optional

from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_openai import AzureChatOpenAI, AzureOpenAIEmbeddings
from langchain_core.output_parsers import StrOutputParser
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_core.documents import Document
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage

from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph.message import add_messages

# 환경변수 로드
load_dotenv('env', override=True)

AZURE_OPENAI_API_KEY = os.getenv('AZURE_OPENAI_API_KEY')
END_POINT = os.getenv('END_POINT')
MODEL_NAME = os.getenv('MODEL_NAME')
AZURE_OPENAI_EMB_API_KEY = os.getenv('AZURE_OPENAI_EMB_API_KEY')
EMB_END_POINT = os.getenv('EMB_END_POINT')
EMB_MODEL_NAME = os.getenv('EMB_MODEL_NAME')

os.environ['LANGCHAIN_API_KEY'] = os.getenv('LANGSMITH_API_KEY')
os.environ['LANGCHAIN_ENDPOINT'] = os.getenv('LANGCHAIN_ENDPOINT')
os.environ['LANGCHAIN_TRACING_V2'] = 'true'
os.environ['LANGCHAIN_PROJECT'] = 'Advanced_RAG_ICT'

print(f"API Key: {AZURE_OPENAI_API_KEY[:10]}...")
print(f"Model: {MODEL_NAME}")
print("✅ 환경 설정 완료!")


## 2. LLM 및 Embedding 모델 초기화


In [ ]:
# LLM 초기화
llm = AzureChatOpenAI(
    api_key=AZURE_OPENAI_API_KEY,
    azure_endpoint=END_POINT,
    azure_deployment=MODEL_NAME,
    api_version="2024-12-01-preview",
    temperature=0.1,  # 낮은 temperature로 일관성 있는 답변
)

# Embedding 모델 초기화
emb = AzureOpenAIEmbeddings(
    model=EMB_MODEL_NAME,
    api_key=AZURE_OPENAI_EMB_API_KEY,
    azure_endpoint=EMB_END_POINT,
    api_version="2024-08-01-preview"
)

print("✅ LLM 및 Embedding 모델 초기화 완료!")


## 3. 문서 로딩 및 벡터 스토어 구축

최신ICT시사용어2025.pdf 파일을 청크로 분할하여 벡터 데이터베이스를 구축합니다.


In [ ]:
def build_vectordb(pdf_pattern: str, chunk_size=600, chunk_overlap=100) -> InMemoryVectorStore:
    """PDF 문서를 로드하고 청크로 나누어 벡터 DB 구축"""
    
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=["\n\n", "\n", ". ", " ", ""]
    )
    
    docs = []
    for path in glob.glob(pdf_pattern):
        print(f"📄 로딩 중: {path}")
        loader = PyMuPDFLoader(path)
        docs.extend(loader.load())
    
    print(f"\n총 {len(docs)}개의 페이지 로드됨")
    
    splits = splitter.split_documents(docs)
    print(f"총 {len(splits)}개의 청크로 분할됨")
    
    vectorstore = InMemoryVectorStore.from_documents(splits, embedding=emb)
    return vectorstore

# 최신ICT시사용어2025.pdf 파일 로드
VSTORE = build_vectordb("pdf/최신ICT시사용어2025.pdf", chunk_size=600, chunk_overlap=100)
print("\n✅ 벡터 스토어 구축 완료!")


## 4. State 정의

LangGraph의 State는 그래프 실행 중 데이터를 관리합니다.


In [ ]:
class RAGState(TypedDict):
    """고급 RAG 시스템의 상태 관리"""
    
    # 입력/출력
    messages: Annotated[List[BaseMessage], add_messages]  # 대화 이력
    question: str  # 현재 질문
    answer: str  # 최종 답변
    
    # 쿼리 처리
    rewritten_query: str  # 재작성된 쿼리
    hyde_answer: str  # HyDE 가상 답변
    multi_queries: List[str]  # 다중 쿼리
    
    # 검색 결과
    retrieved_docs: List[Document]  # 검색된 문서들
    reranked_docs: List[Document]  # 재정렬된 문서들
    context: str  # 최종 컨텍스트
    
    # 품질 관리
    relevance_score: float  # 관련성 점수
    need_retry: bool  # 재시도 필요 여부
    retry_count: int  # 재시도 횟수
    
print("✅ State 정의 완료!")


## 5. 유틸리티 함수


In [ ]:
def last_user_text(messages: List[BaseMessage]) -> str:
    """메시지 리스트에서 마지막 사용자 메시지 추출"""
    for m in reversed(messages):
        if isinstance(m, HumanMessage):
            return m.content
    return messages[-1].content if messages else ""

def format_docs(docs: List[Document]) -> str:
    """문서 리스트를 컨텍스트 문자열로 변환"""
    blocks = []
    for i, d in enumerate(docs, 1):
        src = d.metadata.get("source", "unknown")
        pg = d.metadata.get("page", "?")
        blocks.append(
            f"### 문서 {i} [출처: {os.path.basename(src)} p.{pg}]\n"
            f"{d.page_content}"
        )
    return "\n\n---\n\n".join(blocks)

print("✅ 유틸리티 함수 정의 완료!")


## 6. 노드 구현

### 6.1 Query Rewriting 노드

질문을 검색에 최적화된 형태로 재작성합니다.


In [ ]:
QUERY_REWRITE_PROMPT = ChatPromptTemplate.from_messages([
    ("system",
     "너는 정보 검색 전문가야. 사용자의 질문을 ICT 용어 검색에 최적화된 쿼리로 재작성해.\n"
     "규칙:\n"
     "1) 모호한 표현을 구체적인 ICT 용어로 변환\n"
     "2) 핵심 키워드 보존 및 관련 동의어 추가\n"
     "3) 1~2문장으로 간결하게\n"
     "4) 순수 텍스트만 출력"),
    ("human", "원문 질문: {question}\n재작성 쿼리:")
])

rewrite_chain = QUERY_REWRITE_PROMPT | llm | StrOutputParser()

def node_query_rewrite(state: RAGState) -> dict:
    """질문을 검색에 최적화된 형태로 재작성"""
    q = state.get("question") or last_user_text(state.get("messages", []))
    rewritten = rewrite_chain.invoke({"question": q}).strip() or q
    print(f"🔄 [Query Rewrite] {q} → {rewritten}")
    return {"rewritten_query": rewritten}

print("✅ Query Rewriting 노드 정의 완료!")


### 6.2 HyDE 노드

가상 답변을 생성하여 유사한 문서를 더 잘 찾을 수 있게 합니다.


In [ ]:
HYDE_PROMPT = ChatPromptTemplate.from_messages([
    ("system",
     "너는 ICT 용어 전문가야. 질문에 대한 예상 답변을 간단히 작성해.\n"
     "실제 정보가 없어도 괜찮아. 이 답변은 유사한 문서를 찾는데 사용될 거야."),
    ("human", "질문: {question}\n예상 답변:")
])

hyde_chain = HYDE_PROMPT | llm | StrOutputParser()

def node_hyde(state: RAGState) -> dict:
    """가상 답변 생성 (HyDE)"""
    q = state.get("question") or last_user_text(state.get("messages", []))
    hyde_answer = hyde_chain.invoke({"question": q}).strip()
    print(f"💭 [HyDE] 가상 답변: {hyde_answer[:80]}...")
    return {"hyde_answer": hyde_answer}

print("✅ HyDE 노드 정의 완료!")


### 6.3 Multi-Query 생성 노드

다양한 관점에서 질문을 재구성합니다.


In [ ]:
MULTI_QUERY_PROMPT = ChatPromptTemplate.from_messages([
    ("system",
     "너는 검색 쿼리 전문가야. 원래 질문을 다양한 관점에서 3개의 다른 검색 쿼리로 변환해.\n"
     "각 쿼리는 한 줄로 작성하고, 번호나 기호 없이 줄바꿈으로만 구분해."),
    ("human", "원문 질문: {question}")
])

multi_query_chain = MULTI_QUERY_PROMPT | llm | StrOutputParser()

def node_multi_query(state: RAGState) -> dict:
    """다양한 관점의 쿼리 생성"""
    q = state.get("rewritten_query") or state.get("question") or last_user_text(state.get("messages", []))
    result = multi_query_chain.invoke({"question": q})
    queries = [line.strip() for line in result.split("\n") if line.strip()]
    queries = [q.lstrip("123456789.-) ") for q in queries]  # 번호 제거
    print(f"🔍 [Multi-Query] {len(queries)}개의 쿼리 생성")
    for i, query in enumerate(queries, 1):
        print(f"   {i}. {query}")
    return {"multi_queries": queries}

print("✅ Multi-Query 노드 정의 완료!")


### 6.4 Multi-Source Retrieval 노드

다중 쿼리로 문서를 검색하고 중복을 제거합니다.


In [ ]:
def node_retrieve(state: RAGState) -> dict:
    """다중 쿼리로 문서 검색 및 중복 제거"""
    
    # 검색할 쿼리들 수집
    queries = []
    
    # 1. 재작성 쿼리
    if state.get("rewritten_query"):
        queries.append(state["rewritten_query"])
    
    # 2. HyDE 답변
    if state.get("hyde_answer"):
        queries.append(state["hyde_answer"])
    
    # 3. 다중 쿼리
    if state.get("multi_queries"):
        queries.extend(state["multi_queries"][:2])  # 상위 2개만
    
    # 4. 원본 질문 (fallback)
    if not queries:
        q = state.get("question") or last_user_text(state.get("messages", []))
        queries.append(q)
    
    print(f"\n📚 [Retrieval] {len(queries)}개의 쿼리로 검색 시작")
    
    # 각 쿼리로 검색
    all_docs = []
    seen_content = set()
    
    for i, query in enumerate(queries, 1):
        docs = VSTORE.similarity_search(query, k=5)
        print(f"   쿼리 {i}: {len(docs)}개 문서 검색")
        
        # 중복 제거
        for doc in docs:
            content_hash = hash(doc.page_content)
            if content_hash not in seen_content:
                seen_content.add(content_hash)
                all_docs.append(doc)
    
    print(f"   총 {len(all_docs)}개의 고유 문서 검색됨\n")
    return {"retrieved_docs": all_docs}

print("✅ Multi-Source Retrieval 노드 정의 완료!")


### 6.5 Reranking 노드

검색된 문서를 관련성 기준으로 재정렬하고 "Lost in the Middle" 문제를 방지합니다.


In [ ]:
def node_rerank(state: RAGState) -> dict:
    """검색된 문서를 관련성 기준으로 재정렬"""
    
    docs = state.get("retrieved_docs", [])
    if not docs:
        return {"reranked_docs": []}
    
    q = state.get("question") or last_user_text(state.get("messages", []))
    
    # 쿼리와 각 문서의 유사도 계산
    scored_docs = []
    for doc in docs:
        # similarity_search_with_score를 사용하여 유사도 계산
        results = VSTORE.similarity_search_with_score(doc.page_content, k=1)
        similarity = results[0][1] if results else 999  # 낮을수록 유사
        scored_docs.append((doc, similarity))
    
    # 점수 기준 정렬 (낮을수록 유사)
    scored_docs.sort(key=lambda x: x[1])
    
    # 상위 10개만
    reranked = [doc for doc, _ in scored_docs[:10]]
    
    # Lost in the Middle 방지: Interleave pattern [0, 2, 4, ..., 9, 7, 5, 3, 1]
    final_order = reranked[0::2] + reranked[1::2][::-1]
    
    print(f"🔀 [Rerank] {len(final_order)}개 문서 재정렬 완료")
    return {"reranked_docs": final_order}

print("✅ Reranking 노드 정의 완료!")


### 6.6 Answer Generation 노드

컨텍스트를 기반으로 정확한 답변을 생성합니다.


In [ ]:
ANSWER_PROMPT = ChatPromptTemplate.from_messages([
    ("system",
     "너는 ICT 용어 전문가야. 제공된 컨텍스트를 기반으로 정확하고 상세하게 답변해.\n\n"
     "규칙:\n"
     "1) 컨텍스트에 있는 정보만 사용\n"
     "2) ICT 용어는 정확한 정의와 설명 포함\n"
     "3) 모르면 '해당 정보를 찾을 수 없습니다'라고 명확히 표현\n"
     "4) 답변 마지막에 출처 페이지 명시\n"
     "5) 전문적이면서도 이해하기 쉽게 설명"),
    MessagesPlaceholder("messages"),
    ("human", "질문: {question}\n\n컨텍스트:\n{context}\n\n답변:")
])

answer_chain = ANSWER_PROMPT | llm | StrOutputParser()

def node_answer(state: RAGState) -> dict:
    """컨텍스트 기반 답변 생성"""
    
    docs = state.get("reranked_docs", [])
    if not docs:
        return {"answer": "검색된 문서가 없습니다."}
    
    context = format_docs(docs[:5])  # 상위 5개 문서만 사용
    q = state.get("question") or last_user_text(state.get("messages", []))
    
    answer = answer_chain.invoke({
        "question": q,
        "context": context,
        "messages": state.get("messages", [])
    })
    
    print(f"✍️  [Answer] 답변 생성 완료 (길이: {len(answer)} 문자)")
    return {
        "answer": answer,
        "context": context,
        "messages": [AIMessage(content=answer)]
    }

print("✅ Answer Generation 노드 정의 완료!")


### 6.7 Self-RAG: 품질 평가 노드

생성된 답변의 품질을 평가하고 필요시 재시도를 결정합니다.


In [ ]:
EVAL_PROMPT = ChatPromptTemplate.from_messages([
    ("system",
     "너는 답변 품질 평가자야. 질문과 답변을 보고 품질을 평가해.\n\n"
     "평가 기준:\n"
     "- 질문에 직접적으로 답하는가?\n"
     "- 정보가 충분한가?\n"
     "- 사실에 기반하는가?\n\n"
     "'good' 또는 'bad' 중 하나만 출력해."),
    ("human", "질문: {question}\n\n답변: {answer}\n\n평가:")
])

eval_chain = EVAL_PROMPT | llm | StrOutputParser()

def node_evaluate(state: RAGState) -> dict:
    """답변 품질 평가"""
    
    q = state.get("question") or last_user_text(state.get("messages", []))
    answer = state.get("answer", "")
    
    if not answer or "찾을 수 없습니다" in answer:
        return {"relevance_score": 0.0, "need_retry": True}
    
    eval_result = eval_chain.invoke({
        "question": q,
        "answer": answer
    }).strip().lower()
    
    is_good = "good" in eval_result
    score = 1.0 if is_good else 0.3
    
    retry_count = state.get("retry_count", 0)
    need_retry = (not is_good) and (retry_count < 2)  # 최대 2번까지 재시도
    
    print(f"⚖️  [Evaluate] 품질: {eval_result}, 점수: {score}, 재시도: {need_retry}")
    
    return {
        "relevance_score": score,
        "need_retry": need_retry,
        "retry_count": retry_count + 1 if need_retry else retry_count
    }

print("✅ Self-RAG 평가 노드 정의 완료!")


## 7. LangGraph 구성

모든 노드를 연결하여 고급 RAG 워크플로우를 구성합니다.


In [ ]:
# 조건부 분기 함수
def should_retry(state: RAGState) -> str:
    """재시도 필요 여부 판단"""
    if state.get("need_retry", False):
        print("🔄 [Router] 품질 미달 → 재시도\n")
        return "retry"
    else:
        print("✅ [Router] 품질 양호 → 종료\n")
        return "end"

# 그래프 구성
graph = StateGraph(RAGState)

# 노드 추가
graph.add_node("rewrite", node_query_rewrite)
graph.add_node("hyde", node_hyde)
graph.add_node("multi_query", node_multi_query)
graph.add_node("retrieve", node_retrieve)
graph.add_node("rerank", node_rerank)
graph.add_node("answer", node_answer)
graph.add_node("evaluate", node_evaluate)

# 엣지 연결
graph.add_edge(START, "rewrite")
graph.add_edge(START, "hyde")
graph.add_edge("rewrite", "multi_query")
graph.add_edge("hyde", "retrieve")
graph.add_edge("multi_query", "retrieve")
graph.add_edge("retrieve", "rerank")
graph.add_edge("rerank", "answer")
graph.add_edge("answer", "evaluate")

# 조건부 엣지: 평가 결과에 따라 재시도 or 종료
graph.add_conditional_edges(
    "evaluate",
    should_retry,
    {
        "retry": "multi_query",  # 재시도 시 다른 쿼리로 재검색
        "end": END
    }
)

# 메모리 체크포인터 추가 (대화 이력 관리)
memory = MemorySaver()
app = graph.compile(checkpointer=memory)

print("\n✅ 고급 RAG 그래프 구성 완료!")


## 8. 그래프 시각화


In [ ]:
from IPython.display import Image, display

try:
    display(Image(app.get_graph().draw_mermaid_png()))
except Exception as e:
    print(f"그래프 시각화 실패: {e}")
    print("(Mermaid 렌더링이 지원되지 않는 환경일 수 있습니다)")


## 9. 실행 함수


In [ ]:
def chat(question: str, session_id: str = "default") -> str:
    """RAG 시스템 실행 함수"""
    
    print(f"\n{'='*80}")
    print(f"💬 질문: {question}")
    print(f"{'='*80}\n")
    
    # 초기 상태
    initial_state = {
        "messages": [HumanMessage(content=question)],
        "question": question,
        "retry_count": 0,
        "need_retry": False
    }
    
    # 그래프 실행
    config = {"configurable": {"thread_id": session_id}}
    result = app.invoke(initial_state, config=config)
    
    answer = result.get("answer", "답변을 생성할 수 없습니다.")
    
    print(f"\n{'='*80}")
    print("📝 답변:")
    print(f"{'='*80}")
    print(answer)
    print(f"\n[📊 품질 점수: {result.get('relevance_score', 0):.2f}]")
    print(f"[🔄 재시도 횟수: {result.get('retry_count', 0)}]")
    
    return answer

print("✅ 실행 함수 정의 완료!")


## 10. 테스트 실행

다양한 ICT 용어에 대해 질문해봅니다.


In [ ]:
# 테스트 질문 1: AI 관련
chat("AI 에이전트가 뭐야?")


In [ ]:
# 테스트 질문 2: RAG 관련
chat("RAG에 대해 자세히 설명해줘")


In [ ]:
# 테스트 질문 3: 최신 용어
chat("멀티모달 AI란?")


In [ ]:
# 테스트 질문 4: 기술 비교
chat("LLM과 SLM의 차이점은?")


In [ ]:
# 테스트 질문 5: 응용 질문
chat("프롬프트 엔지니어링 기법에는 어떤 것들이 있어?")


## 11. 스트리밍 모드

각 노드의 실행 과정을 단계별로 확인할 수 있습니다.


In [ ]:
def chat_stream(question: str, session_id: str = "default"):
    """각 노드의 실행 과정을 스트리밍으로 확인"""
    
    print(f"\n{'='*80}")
    print(f"💬 질문: {question}")
    print(f"{'='*80}\n")
    
    initial_state = {
        "messages": [HumanMessage(content=question)],
        "question": question,
        "retry_count": 0
    }
    
    config = {"configurable": {"thread_id": session_id}}
    
    print("🔄 실행 과정:\n")
    for i, step in enumerate(app.stream(initial_state, config=config), 1):
        node_name = list(step.keys())[0]
        print(f"   Step {i}: [{node_name}]")
    
    # 최종 결과
    final = app.invoke(initial_state, config=config)
    print(f"\n{'='*80}")
    print("📝 최종 답변:")
    print(f"{'='*80}")
    print(final.get("answer", "답변 없음"))

# 스트리밍 테스트 (필요시 주석 해제)
# chat_stream("벡터 데이터베이스란?")


## 12. 성능 분석 도구


In [ ]:
import time

def analyze_performance(questions: List[str]):
    """여러 질문에 대한 성능 분석"""
    
    results = []
    
    print(f"\n{'='*80}")
    print("📊 성능 분석 시작")
    print(f"{'='*80}\n")
    
    for q in questions:
        start_time = time.time()
        
        initial_state = {
            "messages": [HumanMessage(content=q)],
            "question": q,
            "retry_count": 0
        }
        
        result = app.invoke(initial_state)
        
        elapsed = time.time() - start_time
        
        results.append({
            "question": q,
            "time": elapsed,
            "score": result.get("relevance_score", 0),
            "retries": result.get("retry_count", 0),
            "answer_length": len(result.get("answer", ""))
        })
        
        print(f"✓ {q[:50]}... | {elapsed:.2f}초 | 점수: {result.get('relevance_score', 0):.2f}")
    
    print(f"\n{'='*80}")
    print("📈 성능 요약")
    print(f"{'='*80}")
    print(f"평균 응답 시간: {sum(r['time'] for r in results) / len(results):.2f}초")
    print(f"평균 품질 점수: {sum(r['score'] for r in results) / len(results):.2f}")
    print(f"총 재시도 횟수: {sum(r['retries'] for r in results)}")
    print(f"평균 답변 길이: {sum(r['answer_length'] for r in results) / len(results):.0f}자")
    
    return results

# 성능 테스트 (필요시 주석 해제)
# test_questions = [
#     "생성형 AI란?",
#     "프롬프트 엔지니어링 기법은?",
#     "벡터 데이터베이스의 용도는?",
#     "파인튜닝과 RAG의 차이점은?"
# ]
# performance_results = analyze_performance(test_questions)


## 13. 추가 개선 아이디어

### 이미 적용된 기법들:
✅ **Query Rewriting** - 검색 최적화  
✅ **HyDE** - 가상 답변 기반 검색  
✅ **Multi-Query** - 다양한 관점에서 검색  
✅ **Reranking** - 결과 재정렬  
✅ **Self-RAG** - 품질 자체 평가  
✅ **Memory Management** - 대화 이력 관리  

### 추가로 적용 가능한 기법들:

#### 1. 검색 개선
- **Hybrid Search**: BM25 + Semantic Search 결합
- **Contextual Compression**: 검색 결과에서 핵심만 추출
- **Parent Document Retriever**: 작은 청크로 검색, 큰 문맥 제공
- **Ensemble Retriever**: 여러 검색 방법 결합

#### 2. 답변 품질 향상
- **Chain-of-Thought**: 단계별 추론 과정
- **Self-Consistency**: 여러 답변 중 가장 일관된 것 선택
- **Citation**: 각 주장에 명확한 출처 표시
- **Verification**: 생성된 답변 사실 검증

#### 3. 평가 고도화
- **RAGAs Framework**: 자동화된 평가
- **User Feedback Loop**: 사용자 피드백 반영
- **A/B Testing**: 다양한 설정 비교
- **Metrics Tracking**: 성능 지표 추적

#### 4. 성능 최적화
- **Caching**: 자주 묻는 질문 캐싱
- **Async Processing**: 비동기 처리
- **Batch Processing**: 배치 처리
- **Vector DB Optimization**: 벡터 DB 최적화 (FAISS, Chroma 등)


## 14. 정리 및 다음 단계

### 🎉 완성된 시스템의 특징:

1. **다층적 쿼리 처리**
   - Query Rewriting으로 검색 최적화
   - HyDE로 유사 문서 검색
   - Multi-Query로 다양한 관점 확보

2. **고급 검색 전략**
   - 다중 소스에서 검색
   - 중복 제거
   - Reranking으로 관련성 향상

3. **품질 보증**
   - Self-RAG로 자체 평가
   - 품질 미달시 자동 재시도
   - 출처 명확히 표시

4. **대화 지원**
   - Memory로 컨텍스트 유지
   - 이전 대화 참조 가능

### 🚀 다음 단계:

1. **프로덕션화**
   - FAISS/Chroma로 벡터 DB 영구 저장
   - FastAPI로 REST API 구축
   - Docker 컨테이너화

2. **UI 추가**
   - Streamlit/Gradio로 웹 인터페이스
   - 실시간 스트리밍 답변
   - 출처 시각화

3. **모니터링**
   - LangSmith로 추적
   - 성능 메트릭 대시보드
   - 사용자 피드백 수집

4. **확장**
   - 더 많은 문서 추가
   - 도메인별 전문화
   - 다국어 지원
